# Capítulo 9: Classificação

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 4 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> A visão geral deste capítulo ainda será escrita.

## Seções

| Seção | Tópico |
|---|---|
| [9.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-por-que-nao-regressao-linear.html) | Por que Não Regressão Linear |
| [9.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-regressao-logistica.html) | Regressão Logística |
| [9.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/03-logistica-multinomial.html) | Logística Multinomial |
| [9.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/04-modelos-generativos.html) | Modelos Generativos: LDA, QDA e Naive Bayes |
| [9.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/05-avaliando-um-classificador.html) | Avaliando um Classificador |
| [9.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/06-comparando-os-metodos.html) | Comparando os Métodos |

## Por que Não Regressão Linear

> **📌 Nota**
>
> Esta seção corresponde às seções 4.1 e 4.2 de James et al. (2023).

O capítulo anterior tratou sempre do mesmo tipo de pergunta: dado um conjunto de preditores, que número a resposta deve assumir — vendas em milhares de unidades, consumo em milhas por galão. `Default` muda o tipo da resposta. Dez mil clientes de cartão de crédito, e a pergunta agora é se um cliente fica inadimplente ou não: `inadimplente` não é uma quantidade, é uma categoria, `sim` ou `não`. `saldo`, `renda` e `estudante` (`sim`/`não`) continuam preditores de sempre; o que muda é o alvo, e é essa mudança que o resto do capítulo resolve.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression

plt.style.use("estilo-figuras.mplstyle")

### O `Default`: dez mil clientes, um alvo qualitativo

In [ ]:
base = pd.read_csv("dados/Default.csv")
base.shape, base.columns.tolist()

Dez mil linhas, quatro colunas: `inadimplente` é o alvo; `estudante` é uma segunda categórica; `saldo` (quanto o cliente deve no cartão) e `renda` (a renda anual do cliente) são numéricas, as duas em dólares.

In [ ]:
n_sim = int((base["inadimplente"] == "sim").sum())
n_nao = int((base["inadimplente"] == "não").sum())
proporcao_sim = n_sim / len(base)

n_sim, n_nao, round(proporcao_sim, 4)

333 dos 10.000 clientes ficam inadimplentes; 9.667, não. A proporção, 0,0333, é baixa: menos de um em trinta. `Default` é um conjunto desequilibrado, e um classificador que sempre responder "não" já acerta a maioria — um ponto que volta na seção 9.5, quando "acerta a maioria" deixa de bastar como medida.

### O saldo separa; a renda, não

In [ ]:
# Figura: `Default`: saldo e renda de 10.000 clientes, com quem ficou inadimplente (`sim`) destacado sobre quem não ficou (`não`). Ao lado, os mesmos dois grupos em caixas — quartil 25%, mediana e quartil 75% — para saldo e para renda.
nao = base[base["inadimplente"] == "não"]
sim = base[base["inadimplente"] == "sim"]

def desenha_caixas(ax, dados_nao, dados_sim, rotulo_y):
    for dados, posicao, cor in [(dados_nao, 1, "C0"), (dados_sim, 2, "C1")]:
        propriedades = dict(color=cor, linewidth=1.6)
        ax.boxplot(
            dados, positions=[posicao], widths=0.5,
            boxprops=propriedades, whiskerprops=propriedades,
            capprops=propriedades, medianprops=propriedades,
            flierprops=dict(markeredgecolor=cor, markersize=3, alpha=0.5),
        )
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["não", "sim"])
    ax.set_xlabel("inadimplente")
    ax.set_ylabel(rotulo_y)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4.4))

ax1.scatter(nao["saldo"], nao["renda"], color="C0", s=10, alpha=0.35, label="não")
ax1.scatter(sim["saldo"], sim["renda"], color="C1", s=14, alpha=0.85, label="sim", zorder=3)
ax1.set_xlabel("saldo (dólares)")
ax1.set_ylabel("renda (dólares)")
ax1.legend(title="inadimplente", loc="upper right", markerscale=1.6)

desenha_caixas(ax2, nao["saldo"], sim["saldo"], "saldo (dólares)")
desenha_caixas(ax3, nao["renda"], sim["renda"], "renda (dólares)")

plt.tight_layout()
plt.show()

A dispersão à esquerda já sugere uma leitura: os pontos laranja (`sim`) se acumulam à direita, em saldos altos; os azuis (`não`), à esquerda. Em renda, as duas cores se misturam ao longo de todo o eixo vertical. As caixas ao lado conferem essa leitura em vez de só ilustrá-la:

In [ ]:
q1_saldo_nao = float(nao["saldo"].quantile(0.25))
q3_saldo_nao = float(nao["saldo"].quantile(0.75))
q1_saldo_sim = float(sim["saldo"].quantile(0.25))
q3_saldo_sim = float(sim["saldo"].quantile(0.75))
sobrepoe_saldo = not (q3_saldo_nao < q1_saldo_sim or q3_saldo_sim < q1_saldo_nao)

q1_renda_nao = float(nao["renda"].quantile(0.25))
q3_renda_nao = float(nao["renda"].quantile(0.75))
q1_renda_sim = float(sim["renda"].quantile(0.25))
q3_renda_sim = float(sim["renda"].quantile(0.75))
sobrepoe_renda = not (q3_renda_nao < q1_renda_sim or q3_renda_sim < q1_renda_nao)

(
    round(q1_saldo_nao, 2), round(q3_saldo_nao, 2), round(q1_saldo_sim, 2), round(q3_saldo_sim, 2), sobrepoe_saldo,
    round(q1_renda_nao, 2), round(q3_renda_nao, 2), round(q1_renda_sim, 2), round(q3_renda_sim, 2), sobrepoe_renda,
)

Em `saldo`, o quartil 75% de quem não ficou inadimplente (1.128,25) fica abaixo do quartil 25% de quem ficou (1.511,61): as duas caixas nem se tocam, `sobrepoe_saldo` sai `False`. Em `renda`, o intervalo de quem não ficou inadimplente vai de 21.405,06 a 43.823,76, e o de quem ficou, de 19.027,51 a 43.067,33 — um contido quase inteiramente dentro do outro, `sobrepoe_renda` sai `True`. O saldo separa os dois grupos; a renda, não.

### Por que a reta não serve para probabilidade

Recodificando `inadimplente` como 0 (`não`) e 1 (`sim`), nada impede de ajustar a mesma `LinearRegression` do capítulo anterior sobre esse alvo — o método não sabe, e não pergunta, se `y` é uma venda em milhares de unidades ou uma categoria disfarçada de número. O que ele devolve, porém, deixa de ser uma venda prevista e passa a ser lido como uma probabilidade prevista: a de que aquele cliente fique inadimplente.

In [ ]:
y = (base["inadimplente"] == "sim").astype(int)
X = base[["saldo"]]

reta = LinearRegression().fit(X, y)
previsoes = reta.predict(X)

n_negativas = int((previsoes < 0).sum())
minimo_previsto = float(previsoes.min())

n_negativas, round(minimo_previsto, 4)

3.123 dos 10.000 clientes recebem da reta uma previsão negativa — quase um em cada três. O mínimo, -0,0752, é a previsão para quem tem o menor saldo do conjunto. Probabilidade negativa não tem leitura possível: nenhum cliente tem chance "menos que zero por cento" de ficar inadimplente, e é esse o problema — não que a reta erre feio de vez em quando, mas que uma fração inteira das suas previsões caia fora do intervalo em que uma probabilidade pode existir.

### A curva que não sai de [0, 1]

In [ ]:
# Figura: Saldo contra a previsão de inadimplência, para os mesmos clientes de `Default`. Esquerda: a reta ajustada acima, que cruza a faixa sombreada — o intervalo [0, 1] — e sai por baixo dela. Direita: uma curva logística ajustada aos mesmos dados; os traços no alto e embaixo marcam, para cada cliente, se ele ficou inadimplente (sim, no topo) ou não (não, embaixo).
logistica = LogisticRegression().fit(X, y)

grade_saldo = pd.DataFrame({"saldo": np.linspace(base["saldo"].min(), base["saldo"].max(), 300)})
reta_grade = reta.predict(grade_saldo)
prob_logistica_grade = logistica.predict_proba(grade_saldo)[:, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)

for ax in (ax1, ax2):
    ax.axhspan(0, 1, color="C2", alpha=0.10)
    ax.plot(nao["saldo"], np.zeros(len(nao)), "|", color="C0", alpha=0.4, markersize=8)
    ax.plot(sim["saldo"], np.ones(len(sim)), "|", color="C1", alpha=0.6, markersize=8)
    ax.set_xlabel("saldo (dólares)")

ax1.plot(grade_saldo["saldo"], reta_grade, color="C3", linewidth=2.2)
ax1.scatter([0.0], [minimo_previsto], color="C3", s=40, zorder=3)
ax1.annotate(
    f"mínimo: {minimo_previsto:.3f}",
    xy=(0.0, minimo_previsto),
    xytext=(14, -6),
    textcoords="offset points",
    fontsize=8,
)
ax1.set_ylabel("previsão / probabilidade de inadimplência")
ax1.set_title("reta")

ax2.plot(grade_saldo["saldo"], prob_logistica_grade, color="C3", linewidth=2.2)
ax2.set_title("curva logística")

plt.tight_layout()
plt.show()

A reta (esquerda) atravessa a faixa sombreada e sai por baixo dela para saldos próximos de zero — a mesma previsão negativa medida acima, agora como curva. A curva logística (direita), ajustada aos mesmos clientes, tem outro formato: um S que se aproxima de 0 e de 1 sem nunca alcançá-los.

In [ ]:
minimo_logistica = float(prob_logistica_grade.min())
maximo_logistica = float(prob_logistica_grade.max())
n_fora_logistica = int(((prob_logistica_grade < 0) | (prob_logistica_grade > 1)).sum())

round(minimo_logistica, 5), round(maximo_logistica, 4), n_fora_logistica

Sobre a mesma grade de 300 valores de saldo usada na figura, a curva logística vai de 0,00002 a 0,9810 — perto das bordas, mas sem cruzá-las — e `n_fora_logistica` conta zero pontos fora de [0, 1]. Como a curva se ajusta e por que ela tem esse formato é assunto da próxima seção; o que importa aqui é só que ela existe, e que resolve o problema que a reta tem.

### Mais de duas classes, e a ordem que a codificação inventa

O problema muda de figura, mas não desaparece, quando o alvo tem mais de duas categorias. Um paciente chega ao pronto-socorro com sintomas que apontam para um de três diagnósticos — derrame, overdose ou convulsão —, e codificar isso como `Y = 1` para derrame, `2` para overdose e `3` para convulsão impõe duas coisas que a lista de diagnósticos não tinha: uma ordem entre os três, e a afirmação de que a distância entre derrame e overdose é a mesma que entre overdose e convulsão. Trocar a ordem — `1` para convulsão, `2` para derrame, `3` para overdose — é uma codificação igualmente válida, e produz um modelo linear diferente do primeiro; nenhuma das duas é mais correta, porque não existe uma escala numérica por trás do diagnóstico que a codificação possa recuperar. Reta ou curva, uma resposta com mais de duas categorias sem ordem natural pede outro tratamento — o que a seção 9.3 faz.

## Regressão Logística

> **📌 Nota**
>
> Esta seção corresponde às seções 4.3.1, 4.3.2, 4.3.3 e 4.3.4 de James et al. (2023).

A seção anterior deixou a curva em S pronta, sem dizer de onde ela vem nem como um coeficiente se lê nela. A regressão logística resolve as duas coisas: ajusta essa curva a dado real, e dá um coeficiente por preditor — só que lido de um jeito diferente do que o capítulo 8 ensinou para a reta.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

plt.style.use("estilo-figuras.mplstyle")

base = pd.read_csv("dados/Default.csv")
y = (base["inadimplente"] == "sim").astype(int)

### A função logística

Para um preditor $X$ e coeficientes $\beta_0$ e $\beta_1$, a regressão logística modela a probabilidade como

$$
p(X) = \frac{e^{\beta_0 + \beta_1 X}}{1 + e^{\beta_0 + \beta_1 X}}.
$$

O denominador é sempre o numerador mais 1, e os dois são positivos para qualquer $\beta_0$, $\beta_1$ e $X$ reais — a fração fica presa em (0, 1) pela forma da conta, não pelo ajuste. É essa propriedade, e não o valor de nenhum coeficiente, que resolve o problema da seção anterior.

In [ ]:
grade_z = np.linspace(-10, 10, 400)
p_z = np.exp(grade_z) / (1 + np.exp(grade_z))

p_extremo_baixo = float(p_z.min())
p_extremo_alto = float(p_z.max())

f"{p_extremo_baixo:.6f}", f"{p_extremo_alto:.6f}"

Em $\beta_0 + \beta_1 X = -10$, a curva vale 0,000045; em $+10$, vale 0,999955 — perto das bordas de [0, 1], mas sem nunca as tocar, por mais que $\beta_0 + \beta_1 X$ se afaste de zero.

In [ ]:
# Figura: A função logística, para z = β0 + β1X entre -10 e 10. A curva se aproxima de 0 e de 1 sem nunca alcançá-los, e é essa forma que sustenta todo ajuste desta seção.
fig, ax = plt.subplots()
ax.axhspan(0, 1, color="C2", alpha=0.10)
ax.plot(grade_z, p_z, color="C0", linewidth=2.2)
ax.set_xlabel("β0 + β1·X")
ax.set_ylabel("p(X)")
plt.tight_layout()
plt.show()

### O ajuste sobre saldo

In [ ]:
modelo_saldo = LogisticRegression(C=np.inf, max_iter=10000).fit(base[["saldo"]], y)

intercepto_saldo = float(modelo_saldo.intercept_[0])
coef_saldo = float(modelo_saldo.coef_[0, 0])

round(intercepto_saldo, 4), round(coef_saldo, 4)

`C=np.inf` desliga a penalização que o `LogisticRegression` aplica por padrão, deixando o ajuste livre para maximizar a verossimilhança sem encolher nenhum coeficiente. Sobre `saldo` sozinho, o ajuste dá intercepto -10,6513 e coeficiente 0,0055.

### A leitura do coeficiente: log-chance, não probabilidade

$$
\log\left(\frac{p(X)}{1-p(X)}\right) = \beta_0 + \beta_1 X.
$$

> **🔷 Conceito**
>
> Um aumento de uma unidade em $X$ soma $\beta_1$ à **log-chance** de $Y = 1$, não à probabilidade $p(X)$. Como a relação entre $p(X)$ e $X$ não é uma reta, o efeito de $X$ sobre a probabilidade depende de onde $X$ já está — diferente da regressão linear do capítulo 8, em que $\beta_1$ valia o mesmo incremento em qualquer ponto.

Sobre `saldo`, cada dólar a mais soma sempre os mesmos 0,0055 à log-chance de inadimplência — mas o efeito sobre a probabilidade não é constante:

In [ ]:
grade_previsao = pd.DataFrame({"saldo": [1000, 2000]})
prob_1000, prob_2000 = modelo_saldo.predict_proba(grade_previsao)[:, 1]

round(float(prob_1000) * 100, 2), round(float(prob_2000) * 100, 2)

Para um saldo de mil dólares, a probabilidade prevista de inadimplência é 0,58%; para um saldo de dois mil dólares — o dobro do primeiro —, ela não dobra: sobe para 58,58%. É essa não linearidade que a curva em S carrega, e que a reta da seção anterior não tinha como representar.

### O paradoxo do estudante

In [ ]:
estudantes = base[base["estudante"] == "sim"]
nao_estudantes = base[base["estudante"] == "não"]

taxa_estudante = float((estudantes["inadimplente"] == "sim").mean())
taxa_nao_estudante = float((nao_estudantes["inadimplente"] == "sim").mean())

round(taxa_estudante * 100, 2), round(taxa_nao_estudante * 100, 2)

Entre estudantes, 4,31% ficam inadimplentes; entre os demais, 2,92% — olhado sozinho, o estudante é o cliente mais arriscado dos dois.

In [ ]:
X_estudante = (base["estudante"] == "sim").astype(int).to_frame("estudante_sim")
modelo_estudante = LogisticRegression(C=np.inf, max_iter=10000).fit(X_estudante, y)

coef_estudante_sozinho = float(modelo_estudante.coef_[0, 0])
round(coef_estudante_sozinho, 4)

Um modelo logístico que usa só `estudante` como preditor confirma essa leitura: o coeficiente sai positivo, 0,4014 — ser estudante soma à log-chance de inadimplência.

In [ ]:
X_multiplo = base[["saldo", "renda"]].copy()
X_multiplo["estudante_sim"] = (base["estudante"] == "sim").astype(int)

modelo_multiplo = LogisticRegression(C=np.inf, max_iter=10000).fit(X_multiplo, y)
coeficientes_multiplo = pd.Series(modelo_multiplo.coef_[0], index=modelo_multiplo.feature_names_in_)
intercepto_multiplo = float(modelo_multiplo.intercept_[0])

(
    round(intercepto_multiplo, 4),
    round(float(coeficientes_multiplo["saldo"]), 6),
    f"{coeficientes_multiplo['renda']:.3e}",
    round(float(coeficientes_multiplo["estudante_sim"]), 4),
)

Juntar `saldo`, `renda` e `estudante` na mesma equação reverte o sinal: o coeficiente de estudante cai para -0,6468. `renda` quase não muda a conta (3,033e-06), mas `saldo` (0,005737) e `estudante` pesam — e o de `estudante` trocou de sinal por completo. É o mesmo tipo de reviravolta que a seção 8.3 viu no coeficiente de jornal: positivo sozinho, outra coisa depois que os preditores certos entram na mesma equação.

In [ ]:
saldo_medio_estudante = float(estudantes["saldo"].mean())
saldo_medio_nao_estudante = float(nao_estudantes["saldo"].mean())

round(saldo_medio_estudante, 2), round(saldo_medio_nao_estudante, 2)

A explicação está no saldo: em média, um estudante deve 987,82 dólares no cartão; um não estudante, 771,77 — 216,05 dólares a menos.

In [ ]:
desvios = X_multiplo.std()
efeito_padronizado = coeficientes_multiplo * desvios

round(float(efeito_padronizado["saldo"]), 4), round(float(efeito_padronizado["renda"]), 4), round(float(efeito_padronizado["estudante_sim"]), 4)

Coeficientes em dólares, em dólares e num 0/1 não se comparam crus; multiplicar cada um pelo desvio-padrão do próprio preditor põe os três na mesma escala — log-chance por um desvio-padrão de variação. Nessa escala, saldo pesa 2,7748, contra 0,0405 de renda e -0,2948 de estudante: é saldo, não estudante, o preditor que mais empurra a inadimplência para cima, e é essa desvantagem que um estudante típico já carrega ao entrar na conta múltipla — o coeficiente isolado de `estudante` não separa os dois efeitos: sozinho, ele confunde o efeito de ser estudante com o efeito de, em média, dever mais.

In [ ]:
modelo_saldo_estudante = LogisticRegression(C=np.inf, max_iter=10000).fit(
    estudantes[["saldo"]], (estudantes["inadimplente"] == "sim").astype(int)
)
modelo_saldo_nao_estudante = LogisticRegression(C=np.inf, max_iter=10000).fit(
    nao_estudantes[["saldo"]], (nao_estudantes["inadimplente"] == "sim").astype(int)
)

grade_saldo = np.linspace(base["saldo"].min(), base["saldo"].max(), 300)
grade_saldo_df = pd.DataFrame({"saldo": grade_saldo})
prob_estudante = modelo_saldo_estudante.predict_proba(grade_saldo_df)[:, 1]
prob_nao_estudante = modelo_saldo_nao_estudante.predict_proba(grade_saldo_df)[:, 1]
estudante_sempre_abaixo = bool((prob_estudante < prob_nao_estudante).all())

faixa_comum_min = float(max(estudantes["saldo"].min(), nao_estudantes["saldo"].min()))
faixa_comum_max = float(min(estudantes["saldo"].max(), nao_estudantes["saldo"].max()))
grade_comum_df = pd.DataFrame({"saldo": np.linspace(faixa_comum_min, faixa_comum_max, 300)})
prob_estudante_comum = modelo_saldo_estudante.predict_proba(grade_comum_df)[:, 1]
prob_nao_estudante_comum = modelo_saldo_nao_estudante.predict_proba(grade_comum_df)[:, 1]
abaixo_na_faixa_comum = bool((prob_estudante_comum < prob_nao_estudante_comum).all())

base_decis = base.copy()
base_decis["decil_saldo"] = pd.qcut(base_decis["saldo"], 10)
taxas_por_decil = (
    base_decis.groupby(["decil_saldo", "estudante"], observed=True)["inadimplente"]
    .apply(lambda s: (s == "sim").mean())
    .unstack("estudante")
)
estudante_nunca_acima_no_decil = bool((taxas_por_decil["sim"] <= taxas_por_decil["não"]).all())

(
    estudante_sempre_abaixo,
    round(faixa_comum_min, 2),
    round(faixa_comum_max, 2),
    abaixo_na_faixa_comum,
    estudante_nunca_acima_no_decil,
)

Dois modelos logísticos ajustados separadamente — um só com os estudantes, outro só com os não estudantes, cada um usando `saldo` como único preditor — mostram a curva do estudante abaixo da do não estudante em toda a grade: `estudante_sempre_abaixo` sai `True`. O mesmo vale restrito à faixa de saldo que os dois grupos de fato ocupam, de 0,00 a 2.499,02 dólares (`abaixo_na_faixa_comum`). O dado cru confirma isso sem depender de modelo nenhum: dividindo `saldo` em dez decis, a taxa observada de inadimplência do estudante não passa a do não estudante em nenhum dos dez (`estudante_nunca_acima_no_decil`).

In [ ]:
# Figura: Esquerda: taxa de inadimplência contra saldo. As curvas cheias são estimadas por um modelo logístico ajustado separadamente em cada grupo (laranja: estudante; azul: não estudante); as tracejadas marcam a taxa bruta observada de cada grupo, sem olhar para saldo. Direita: distribuição de saldo por grupo — o estudante se concentra em saldos mais altos.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))

ax1.plot(grade_saldo, prob_nao_estudante, color="C0", linewidth=2.2, label="não")
ax1.plot(grade_saldo, prob_estudante, color="C1", linewidth=2.2, label="sim")
ax1.axhline(taxa_nao_estudante, color="C0", linestyle="--", linewidth=1.2, alpha=0.7)
ax1.axhline(taxa_estudante, color="C1", linestyle="--", linewidth=1.2, alpha=0.7)
ax1.annotate(
    "taxa bruta: não", xy=(grade_saldo[-1], taxa_nao_estudante),
    xytext=(-98, -10), textcoords="offset points", fontsize=8, color="C0",
)
ax1.annotate(
    "taxa bruta: sim", xy=(grade_saldo[-1], taxa_estudante),
    xytext=(-98, 6), textcoords="offset points", fontsize=8, color="C1",
)
ax1.set_xlabel("saldo (dólares)")
ax1.set_ylabel("taxa de inadimplência")
ax1.legend(title="estudante", loc="upper left")

for dados, posicao, cor in [(nao_estudantes["saldo"], 1, "C0"), (estudantes["saldo"], 2, "C1")]:
    propriedades = dict(color=cor, linewidth=1.6)
    ax2.boxplot(
        dados, positions=[posicao], widths=0.5,
        boxprops=propriedades, whiskerprops=propriedades,
        capprops=propriedades, medianprops=propriedades,
        flierprops=dict(markeredgecolor=cor, markersize=3, alpha=0.5),
    )
ax2.set_xticks([1, 2])
ax2.set_xticklabels(["não", "sim"])
ax2.set_xlabel("estudante")
ax2.set_ylabel("saldo (dólares)")

plt.tight_layout()
plt.show()

A curva da esquerda e os decis contam a mesma história: em qualquer saldo, o estudante não é o cliente mais arriscado — é a curva laranja que fica embaixo, do primeiro ao último ponto da grade. As linhas tracejadas, que ignoram saldo e mostram só a taxa bruta de cada grupo, invertem essa ordem porque o estudante se concentra em saldos mais altos — a diferença de 216,05 dólares na média medida acima — puxando a média geral dele para cima mesmo com cada saldo individual sendo mais seguro.

In [ ]:
prob_multiplo_todas = modelo_multiplo.predict_proba(X_multiplo)[:, 1]
mascara_estudante = X_multiplo["estudante_sim"] == 1

media_prob_estudante = float(prob_multiplo_todas[mascara_estudante].mean())
media_prob_nao_estudante = float(prob_multiplo_todas[~mascara_estudante].mean())

pd.DataFrame(
    {
        "prevista (média)": [media_prob_estudante, media_prob_nao_estudante],
        "bruta": [taxa_estudante, taxa_nao_estudante],
    },
    index=["estudante", "não estudante"],
).round(6)

A média da probabilidade que o modelo múltiplo prevê para cada estudante, sobre o saldo e a renda que ele de fato tem, é 0,043139; a mesma média entre não estudantes é 0,029195 — e a tabela põe as duas ao lado da taxa bruta de cada grupo, iguais até a sexta casa decimal.

## Logística Multinomial

> **📌 Nota**
>
> Esta seção corresponde à seção 4.3.5 de James et al. (2023).

A seção anterior ajustou uma curva logística para um alvo de duas classes, inadimplente ou não. `origem`, em `Auto`, tem três — americano, europeu e japonês —, e a curva em S da seção 9.2 não responde a uma pergunta de três respostas: ela devolve a probabilidade de uma classe contra a outra, e aqui sobra sempre uma terceira sem lugar nessa conta. É essa extensão, de duas classes para $K$, que a logística multinomial resolve.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

plt.style.use("estilo-figuras.mplstyle")

### A base, de novo

A saída é a mesma da seção 8.4: escolhe-se uma classe para servir de referência — a **base** —, e cada uma das outras ganha o seu próprio conjunto de coeficientes, lidos contra essa base. Com $K$ classes e uma base fixada, restam $K - 1$ conjuntos de coeficientes; para um preditor $X = (x_1, \ldots, x_p)$, a probabilidade de uma classe $k$ diferente da base, e da própria base, ficam

$$
\Pr(Y = k \mid X = x) = \frac{e^{\beta_{k0} + \beta_{k1} x_1 + \cdots + \beta_{kp} x_p}}{1 + \sum_{l \neq \text{base}} e^{\beta_{l0} + \beta_{l1} x_1 + \cdots + \beta_{lp} x_p}}, \qquad
\Pr(Y = \text{base} \mid X = x) = \frac{1}{1 + \sum_{l \neq \text{base}} e^{\beta_{l0} + \cdots + \beta_{lp} x_p}}.
$$

> **🔷 Conceito**
>
> Com $K$ categorias sem ordem natural, escolhe-se uma como base; as $K - 1$ restantes ganham coeficientes próprios, lidos como log-chance contra essa base — o mesmo gesto da seção 8.4, em que `drop_first` descartava uma categoria e as indicadoras liam-se contra ela.

O `scikit-learn`, porém, não ajusta esse formato. Ele usa uma parametrização diferente, chamada *softmax*, que trata as $K$ classes de forma simétrica — nenhuma vira base, e todas ganham o próprio conjunto de coeficientes:

$$
\Pr(Y = k \mid X = x) = \frac{e^{\beta_{k0} + \beta_{k1} x_1 + \cdots + \beta_{kp} x_p}}{\sum_{l=1}^{K} e^{\beta_{l0} + \beta_{l1} x_1 + \cdots + \beta_{lp} x_p}}, \qquad k = 1, \ldots, K.
$$

Essa forma tem $K$ conjuntos de coeficientes, não $K - 1$: é sobreparametrizada, porque somar a mesma constante a $\beta_{k0}, \beta_{k1}, \ldots, \beta_{kp}$ em toda classe $k$ não muda probabilidade nenhuma — a constante aparece em todo expoente do numerador e do denominador, e cancela na divisão. É essa sobra que a parametrização com base elimina, zerando de propósito a linha da classe escolhida; a softmax simplesmente deixa a sobra aí, espalhada pelas $K$ linhas. As duas formas dizem a mesma coisa, e o resto da seção mostra isso com o `Auto`, no lugar do exemplo do pronto-socorro que o livro-texto usa aqui.

### Três origens, cinco preditores

In [ ]:
auto = pd.read_csv("dados/Auto.csv")
n_total = len(auto)

auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
n_com_interrogacao = int(auto["potencia"].isna().sum())

auto_limpo = auto.dropna(subset=["potencia"]).copy()
n_linhas = len(auto_limpo)

n_total, n_com_interrogacao, n_linhas

`dados/Auto.csv` tem 397 carros. `potencia` chega como texto, porque cinco linhas trazem `?` em vez de um número; `pd.to_numeric(auto["potencia"], errors="coerce")` converte a coluna para número e transforma esses cinco `?` em ausente, e `dropna()` descarta as cinco linhas ausentes — sobram 392.

In [ ]:
contagem_origem = auto_limpo["origem"].value_counts().sort_index()
contagem_origem

Das 392 linhas restantes, 245 são americanos (`origem` 1), 68 são europeus (`origem` 2) e 79 são japoneses (`origem` 3). 245 dos 392 carros — 62,5% do conjunto (245/392) — vêm dos Estados Unidos; as outras duas origens dividem o resto. Cinco preditores técnicos — `cilindrada`, `potencia`, `peso`, `aceleracao` e `ano` — tentam prever qual das três é a origem de cada carro.

### Uma linha de coeficientes por classe

In [ ]:
preditores = ["cilindrada", "potencia", "peso", "aceleracao", "ano"]
X = auto_limpo[preditores]
y = auto_limpo["origem"]

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    modelo = LogisticRegression(C=np.inf, max_iter=10000).fit(X, y)

n_iteracoes = int(modelo.n_iter_[0])
n_avisos = len(avisos)
acuracia = float(modelo.score(X, y))

n_iteracoes, n_avisos, round(acuracia, 4)

O ajuste, com o mesmo `C=np.inf` e `max_iter=10000` da seção 9.2, converge em 4.739 iterações; capturando todo aviso emitido durante o `fit`, `n_avisos` sai 0 — nenhum aviso de convergência, nem de nenhum outro tipo. Sobre as mesmas 392 linhas que o treinaram, o modelo acerta 0,7883 das previsões; não é desempenho em dado novo, só a fração de acerto sobre o dado que ele já viu — julgar um classificador em dado que ele não viu é assunto da seção 9.5.

In [ ]:
tabela_coeficientes = pd.DataFrame(
    modelo.coef_, index=pd.Index(modelo.classes_, name="origem"), columns=modelo.feature_names_in_
)
tabela_coeficientes.insert(0, "intercepto", modelo.intercept_)
tabela_coeficientes.round(4)

`coef_` vem com três linhas, não duas: `classes_` dá 1, 2 e 3, a mesma codificação de `origem`, e cada linha traz um coeficiente por coluna de `feature_names_in_`. Nenhuma das três é a base — é a parametrização softmax descrita acima, simétrica, que o `scikit-learn` ajusta por padrão sempre que o alvo tem mais de duas classes.

### A base muda, a previsão não

In [ ]:
classe_base = int(modelo.classes_[0])

coef_recentrado = modelo.coef_ - modelo.coef_[0]
intercepto_recentrado = modelo.intercept_ - modelo.intercept_[0]

tabela_recentrada = pd.DataFrame(
    coef_recentrado, index=pd.Index(modelo.classes_, name="origem"), columns=modelo.feature_names_in_
)
tabela_recentrada.insert(0, "intercepto", intercepto_recentrado)

tabela_lado_a_lado = pd.concat(
    {
        "parametrização softmax": tabela_coeficientes,
        f"contra a base (origem {classe_base})": tabela_recentrada,
    },
    axis=1,
)
tabela_lado_a_lado.round(4)

Subtrair a linha da origem 1 de toda linha de `coef_` — e o mesmo em `intercept_` — produz exatamente a forma com base descrita no começo desta seção: a linha da origem 1 zera por completo, e as outras duas passam a ler-se contra ela. É de novo o gesto do `drop_first` da seção 8.4, agora sobre três classes em vez de duas categorias, e sem reajustar modelo nenhum — a tabela acima só reorganiza os mesmos coeficientes que `ajuste-multinomial` já tinha produzido.

In [ ]:
logitos_recentrados = X.values @ coef_recentrado.T + intercepto_recentrado
exp_logitos = np.exp(logitos_recentrados)
probabilidades_manuais = exp_logitos / exp_logitos.sum(axis=1, keepdims=True)

probabilidades_sklearn = modelo.predict_proba(X)
diferenca_maxima = float(np.abs(probabilidades_manuais - probabilidades_sklearn).max())
probabilidades_batem = bool(np.allclose(probabilidades_manuais, probabilidades_sklearn))

previsao_manual = modelo.classes_[probabilidades_manuais.argmax(axis=1)]
previsao_sklearn = modelo.predict(X)
previsoes_batem = bool(np.array_equal(previsao_manual, previsao_sklearn))

f"{diferenca_maxima:.2e}", probabilidades_batem, previsoes_batem

Recalculando a probabilidade à mão a partir desses logitos recentrados — a exponencial de cada logito dividida pela soma da linha, a própria definição de softmax — e comparando com o `predict_proba` do ajuste original: a diferença máxima, sobre as 392 × 3 entradas, é 1,11e-15, e `probabilidades_batem` sai `True`. A previsão — a classe de maior probabilidade em cada linha — também bate: `previsoes_batem` sai `True`. Os coeficientes mudam de número; o que o modelo prevê para cada um dos 392 carros, não.

### Três classes não são uma escala

A seção 9.1 mostrou esse problema do lado da codificação: rotular derrame, overdose e convulsão como 1, 2 e 3 inventa uma ordem e uma distância que a lista de diagnósticos não tinha. A matriz de confusão do ajuste sobre `Auto` mostra o mesmo problema do lado do erro.

In [ ]:
y_previsto = modelo.predict(X)
matriz = confusion_matrix(y, y_previsto, labels=modelo.classes_)

matriz_rotulada = pd.DataFrame(
    matriz,
    index=pd.Index(modelo.classes_, name="origem verdadeira"),
    columns=pd.Index(modelo.classes_, name="origem prevista"),
)
matriz_rotulada

`confusion_matrix(y, y_previsto)` do `scikit-learn` devolve linha = origem verdadeira, coluna = origem prevista — por isso a tabela acima, e a figura a seguir, rotulam os dois eixos em vez de deixar por conta da memória.

In [ ]:
erros_1_2 = int(matriz[0, 1] + matriz[1, 0])
erros_2_3 = int(matriz[1, 2] + matriz[2, 1])
erros_1_3 = int(matriz[0, 2] + matriz[2, 0])

longe_supera_1_2 = bool(erros_1_3 > erros_1_2)

erros_1_2, erros_2_3, erros_1_3, longe_supera_1_2

Contagem crua, porém, não controla pelo tamanho de cada origem — 245, 68 e 79 carros, a contagem por origem de acima —, e um par que reúne mais carros tem mais chance de acumular erro só por isso.

In [ ]:
n_1_2 = int(contagem_origem[1] + contagem_origem[2])
n_2_3 = int(contagem_origem[2] + contagem_origem[3])
n_1_3 = int(contagem_origem[1] + contagem_origem[3])

taxa_1_2 = erros_1_2 / n_1_2
taxa_2_3 = erros_2_3 / n_2_3
taxa_1_3 = erros_1_3 / n_1_3

longe_supera_1_2_em_taxa = bool(taxa_1_3 > taxa_1_2)

round(taxa_1_2 * 100, 2), round(taxa_2_3 * 100, 2), round(taxa_1_3 * 100, 2), longe_supera_1_2_em_taxa

In [ ]:
# Figura: Matriz de confusão do ajuste sobre as 392 linhas de `Auto`, como bolhas: a área de cada bolha cresce com a raiz quadrada da contagem da célula (maior para célula mais frequente, sem ser proporcional a ela), e o número exato está escrito dentro dela. Azul marca acerto (a diagonal); verde marca erro entre origens vizinhas na numeração (1-2 ou 2-3); laranja marca erro entre os dois extremos da numeração (1-3). Eixo vertical: origem verdadeira; eixo horizontal: origem prevista.
rotulos = ["americano", "europeu", "japonês"]

grupos = {
    "acerto": ([], [], [], "C0"),
    "vizinha na numeração (1-2 ou 2-3)": ([], [], [], "C2"),
    "extremos da numeração (1-3)": ([], [], [], "C1"),
}
for i in range(3):
    for j in range(3):
        contagem = int(matriz[i, j])
        if i == j:
            chave = "acerto"
        elif abs(i - j) == 1:
            chave = "vizinha na numeração (1-2 ou 2-3)"
        else:
            chave = "extremos da numeração (1-3)"
        grupos[chave][0].append(j)
        grupos[chave][1].append(i)
        grupos[chave][2].append(contagem)

fig, ax = plt.subplots(figsize=(6.2, 5.4))
for rotulo_legenda, (xs, ys, contagens, cor) in grupos.items():
    tamanhos = [70 * np.sqrt(c) for c in contagens]
    ax.scatter(xs, ys, s=tamanhos, color=cor, zorder=3)
    for x_pos, y_pos, c in zip(xs, ys, contagens):
        ax.annotate(str(c), xy=(x_pos, y_pos), ha="center", va="center", fontsize=9, color="white", zorder=4)

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(rotulos)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(rotulos)
ax.set_xlabel("origem prevista")
ax.set_ylabel("origem verdadeira")
ax.set_xlim(-0.6, 2.6)
ax.set_ylim(2.6, -0.6)

# Marcadores da legenda em tamanho fixo: nesta figura, só a cor identifica a
# categoria — o tamanho de cada bolha do gráfico já está ocupado codificando
# a contagem daquela célula.
marcadores_legenda = [
    Line2D([], [], marker="o", linestyle="None", color=cor, markersize=9)
    for _, _, _, cor in grupos.values()
]
ax.legend(
    marcadores_legenda, list(grupos.keys()),
    title="leitura da célula", loc="upper center", bbox_to_anchor=(0.5, -0.18),
)
plt.tight_layout()
plt.show()

Se `origem` fosse mesmo uma escala — 1, depois 2, depois 3 —, o par de extremos (1-3) devia ser o mais raro de confundir, não um dos mais comuns. A contagem crua já aponta nessa direção: entre americano e japonês (1-3) o modelo erra 33 vezes (19 + 14), mais que entre americano e europeu (1-2), 17 vezes (7 + 10) — `longe_supera_1_2` sai `True`. Mas as três origens têm tamanhos bem diferentes (245, 68 e 79 carros), e um par que reúne mais carros acumula mais chance de erro só por isso, sem que a numeração tenha nada a ver. Dividindo o erro de cada par pelo total de carros que ele reúne — a taxa de confusão do par —, o quadro muda de escala mas não de conclusão: o par vizinho 1-2 erra em 5,43% dos casos, o par vizinho 2-3 em 22,45%, e o par de extremos, 1-3, em 10,19% — acima da taxa do par vizinho 1-2 (`longe_supera_1_2_em_taxa` sai `True`). O par mais distante na numeração de `origem` não é o mais fácil de separar: a numeração não é a régua que o erro segue.

## Modelos Generativos: LDA, QDA e Naive Bayes

> **📌 Nota**
>
> Esta seção corresponde à seção 4.4 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Avaliando um Classificador

> **📌 Nota**
>
> Esta seção corresponde à seção 4.4.2 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Comparando os Métodos

> **📌 Nota**
>
> Esta seção corresponde à seção 4.5 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Leituras adicionais

*A escrever.*

## Referências

- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.